# Lab 3 — Hybrid Search in ES|QL: FORK, FUSE, Filtering, and Reranking

**Goal:** Combine BM25 + semantic into a single retriever that wins on all the query types that broke each method individually in Lab 2 — expressed in ES|QL with `FORK` (run both searches) and `FUSE` (combine their rankings).

## What you'll learn
- **`FORK ... | FUSE`** — run a BM25 branch and a semantic branch in parallel, then fuse them with Reciprocal Rank Fusion (RRF). Zero score normalization, zero tuning.
- **`FUSE LINEAR`** — weighted score fusion with MinMax normalization, and why the "obvious" weight can backfire.
- **Filtering inside FORK branches** — scope retrieval with a `WHERE` term filter.
- **`RERANK`** — a second-pass precision stage on top of FUSE recall.

## The mental model
```esql
FROM index METADATA _score, _id, _index
| FORK ( <BM25 branch>     | SORT _score DESC | LIMIT 50 )
       ( <semantic branch> | SORT _score DESC | LIMIT 50 )
| FUSE                       -- combine the two ranked lists (RRF by default)
| SORT _score DESC | LIMIT 5
```
Each FORK branch is its own mini-pipeline producing a ranked, LIMITed candidate list. `FUSE` merges branches on `_id` and recomputes `_score`. This returns the **same ranking** as the `_search` `rrf` retriever — it's literally the query the Lab 4 Agent Builder tool uses.

In [ ]:
# --- Workshop helpers (inline — same block across all ES|QL notebooks) ---
# ES|QL edition: every search runs through es.esql.query() instead of es.search().
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

# request_timeout=120: RERANK and COMPLETION (Labs 4-5) call inference per row and
# can take several seconds — the default 10s would time out the LLM step.
es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=120)

def esql(query, **params):
    """Run an ES|QL query with named parameters (?name in the query string).

    Usage:  esql(QUERY, q="securing cluster traffic")
    ES|QL named params take the form params=[{"name": value}, ...]. If your pinned
    client rejects named params, switch to positional `?` and params=[value, ...] —
    never f-string the query text in (injection + teaches the wrong pattern).
    """
    param_list = [{k: v} for k, v in params.items()] if params else None
    return es.esql.query(query=query, params=param_list, format="json")

def rows(resp):
    """Turn an ES|QL response ({columns, values}) into a list of dicts keyed by column."""
    cols = [c["name"] for c in resp["columns"]]
    return [dict(zip(cols, vals)) for vals in resp["values"]]

def show_esql(resp, fields=("id", "title", "summary"), score=True):
    """Pretty-print ES|QL rows as a ranked table (mirrors the DSL notebooks' show_hits)."""
    data = rows(resp)
    if not data:
        print("  (no rows)"); return
    for rank, r in enumerate(data, 1):
        cols = "  ".join(str(r.get(f, "")) for f in fields)
        sc = r.get("_score")
        s = f"  {sc:.4f}" if score and sc is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

print("✓ ES|QL helpers loaded")


# Lab 3 query templates.
Q_SEMANTIC = ("FROM aiewf-workshop-docs METADATA _score\n"
              "| WHERE MATCH(body_semantic, ?q)\n"
              "| SORT _score DESC | LIMIT 60 | KEEP id, title, summary, _score")

Q_BM25 = ('FROM aiewf-workshop-docs METADATA _score\n'
          '| WHERE MATCH(title, ?q, {"boost": 3.0}) OR MATCH(body, ?q)\n'
          '| SORT _score DESC | LIMIT 60 | KEEP id, title, summary, _score')

Q_RRF = ("FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
         "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
         "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
         "| FUSE\n"
         "| SORT _score DESC | LIMIT 60 | KEEP id, title, summary, _score")

print("✓ Lab 3 helpers loaded")

In [ ]:
resp = esql("FROM aiewf-workshop-docs | STATS docs = COUNT(*)")
print(f"Connected to ES {es.info()['version']['number']} | {rows(resp)[0]['docs']} docs")

## RRF — Reciprocal Rank Fusion via `FORK ... | FUSE`

Reciprocal Rank Fusion combines two ranked lists without comparing or normalizing their scores. For each document it computes `1 / (rank_constant + rank)` from each branch and sums them. A doc that ranks #1 in both branches wins; a doc ranked #2 in one and absent from the other still beats a doc that only appears once at #10. **Rank position is all that matters — not raw scores.** That's why RRF needs no normalization and no tuning.

In Lab 2, each retriever failed on a *different* query. Watch RRF rescue all of them — same pipeline, just change the query string in both branches.

In [ ]:
# RRF over the four Lab 2 trap queries. Each one broke a single retriever;
# FUSE lets whichever branch was RIGHT carry the fused ranking.
for q in ["notify me when something goes wrong",   # BM25 buried it
          "8.18 breaking changes",                  # BM25 wrong doc
          "new_primaries",                           # semantic wrong doc
          "exit code 137"]:                          # semantic near-tie
    print(f"\n{'='*60}\nQUERY: {q!r}")
    show_esql(esql(Q_RRF, q=q))

## Prove the win objectively — rank of the known-good doc

Eyeballing the #1 result is not measurement. Build a small judgment set — each query paired with the document a human says is correct — and report the **rank** of that doc under BM25, semantic, and RRF. (We use *rank*, not Recall@5: in a 62-doc corpus a "losing" retriever often still squeaks the target into the top 5, so Recall@5 ≈ 1.0 everywhere and hides the contrast.)

In [ ]:
JUDGMENTS = [
    ("exit code 137",                       "doc-007", "exact id — semantic blurs"),
    ("new_primaries",                       "doc-008", "bare value — semantic wrong doc"),
    ("8.18 breaking changes",               "doc-057", "version — BM25 wrong doc"),
    ("notify me when something goes wrong", "doc-049", "paraphrase — BM25 buries"),
]

# rank_of() runs an ES|QL query and finds the position of the known-good id.
def rank_of(template, query, good_id):
    ids = [r["id"] for r in rows(esql(template, q=query))]
    return ids.index(good_id) + 1 if good_id in ids else None

strategies = {"BM25": Q_BM25, "Semantic": Q_SEMANTIC, "RRF hybrid": Q_RRF}
ranks = {name: [rank_of(t, q, gid) for q, gid, _ in JUDGMENTS] for name, t in strategies.items()}
fmt = lambda r: "—" if r is None else str(r)

print(f"{'Query':<38} {'BM25':>6} {'Semantic':>9} {'RRF':>6}   target")
print("-" * 78)
for i, (query, good_id, note) in enumerate(JUDGMENTS):
    win = "✅" if ranks["RRF hybrid"][i] == 1 else "  "
    print(f"{query:<38} {fmt(ranks['BM25'][i]):>6} {fmt(ranks['Semantic'][i]):>9} "
          f"{fmt(ranks['RRF hybrid'][i]):>6} {win} {good_id}")
print("-" * 78)
print("Rank of the correct doc (1 = perfect). RRF should land #1 on every row,")
print("even where BM25 or Semantic mis-ranked the target.")

## Filtering — scoping retrieval with metadata, inside the FORK branches

To restrict hybrid search to a subset (say, only docs tagged `8.18`), add a `WHERE` term filter to **each** branch. In ES|QL a metadata filter is just another `WHERE` predicate `AND`-ed with the `MATCH`:

```esql
FROM aiewf-workshop-docs METADATA _score, _id, _index
| FORK ( WHERE version_tags == "8.18" AND MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )
       ( WHERE version_tags == "8.18" AND MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )
| FUSE | SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score, version_tags
```

In [ ]:
Q_RRF_FILTERED = (
    "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
    '| FORK ( WHERE version_tags == "8.18" AND MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n'
    '       ( WHERE version_tags == "8.18" AND MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n'
    "| FUSE | SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score, version_tags"
)
print("Filtered to version_tags == '8.18':\n")
show_esql(esql(Q_RRF_FILTERED, q="breaking changes"),
          fields=("id", "title", "summary", "version_tags"))
print("\nOnly 8.18-tagged docs are eligible — the boosted-title distractor (doc-006) is filtered out.")

## Linear combination with MinMax normalization — `FUSE LINEAR`

RRF ignores raw scores. **Linear** fusion uses them — but first normalizes each branch's scores to a 0–1 range with MinMax so they're comparable, then applies per-branch weights:

```esql
| FUSE LINEAR WITH {"weights": {"fork1": 0.5, "fork2": 0.5}, "normalizer": "minmax"}
```
`fork1` is the first FORK branch (BM25 here), `fork2` the second (semantic). Why normalize? **BM25 scores run large; semantic scores run ~0–1.** Without MinMax, BM25 would dominate every sum regardless of weight. `minmax` rescales each branch so the weights — not the raw score magnitudes — drive the blend.

We'll test it on the **paraphrase** query `notify me when something goes wrong`, where the right answer (the Watcher doc, `doc-049`) is one BM25 buries. Watch the weight you choose change whether the correct doc surfaces.

> ⚠️ **Two ES|QL specifics worth knowing:** (1) weights must be **positive** — a weight of `0.0` is rejected. (2) `FUSE LINEAR` MinMax normalizes within each branch's *candidate set* (the top-50 from each FORK arm) and re-normalizes the fused score, so behavior differs from the `_search` `linear` retriever — don't expect identical numbers. The *lesson* — weights are workload-specific and go stale — holds either way.

In [ ]:
def q_linear(w_bm25, w_sem):
    return (
        "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
        "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
        "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
        '| FUSE LINEAR WITH {"weights": {"fork1": %s, "fork2": %s}, "normalizer": "minmax"}\n'
        "| SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score"
        % (w_bm25, w_sem)
    )

# Paraphrase query: the Watcher doc (doc-049) is the right answer. BM25 buries it
# (Lab 2), so leaning the weights toward BM25 drags the wrong docs up.
q = "notify me when something goes wrong"   # target doc-049
print(f"QUERY: {q!r}\n")
print("--- 0.8 BM25 / 0.2 semantic (lean on BM25) ---")
show_esql(esql(q_linear(0.8, 0.2), q=q))
print("\n--- 0.3 BM25 / 0.7 semantic (lean on semantic) ---")
show_esql(esql(q_linear(0.3, 0.7), q=q))
print("\nLeaning BM25 sinks the correct Watcher doc (doc-049) — a lexical distractor wins.")
print("Leaning semantic pulls doc-049 to #1. The 'obvious' lexical lean was exactly wrong here,")
print("and the right weight depends on the query — which is why RRF (no weights) is the default.")
# Note: ES|QL FUSE LINEAR weights must be POSITIVE (a weight of 0.0 is rejected).

## "Just measure the right weights" — OK, let's actually measure them

Sweep the BM25↔semantic balance across the full range and score each split by **MRR** (Mean Reciprocal Rank) over the judgment set. Compare the best linear split against zero-tuning RRF.

In [ ]:
def mrr(template):
    total = 0.0
    for query, good_id, _ in JUDGMENTS:
        r = rank_of(template, query, good_id)
        total += (1.0 / r) if r else 0.0
    return total / len(JUDGMENTS)

print("Baselines (MRR — 1.0 = every target at rank 1):")
print(f"  BM25 only:       {mrr(Q_BM25):.3f}")
print(f"  Semantic only:   {mrr(Q_SEMANTIC):.3f}")
print(f"  RRF (no tuning): {mrr(Q_RRF):.3f}")

print("\nLinear weight sweep (weights must be positive, so 0.1–0.9):")
print(f"  {'BM25':>5} {'semantic':>9} {'MRR':>7}")
print("  " + "-" * 23)
sweep = []
for i in range(1, 10):                       # 0.1 .. 0.9 — FUSE LINEAR rejects a 0.0 weight
    w_sem = round(i * 0.1, 1); w_bm25 = round(1.0 - w_sem, 1)
    sweep.append((w_bm25, w_sem, mrr(q_linear(w_bm25, w_sem))))
best = max(sweep, key=lambda x: x[2])
for w_bm25, w_sem, score in sweep:
    flag = "  <- best measured" if (w_bm25, w_sem) == best[:2] else ""
    print(f"  {w_bm25:>5} {w_sem:>9} {score:>7.3f}{flag}")

print("\n" + "=" * 52)
print(f"Best linear (measured): BM25={best[0]} / semantic={best[1]}  → MRR {best[2]:.3f}")
print(f"RRF, zero tuning:                              → MRR {mrr(Q_RRF):.3f}")
print("=" * 52)
print("\nThe best linear weight leans semantic because THIS judgment set is paraphrase-")
print("and version-heavy. Change the query mix or re-embed and that number moves. RRF")
print("matched it with nothing to tune and nothing to re-calibrate later.")

## The whole story in one picture — strategies × queries

A heatmap of the correct doc's rank for every (strategy, query). Green = rank 1, red = mis-ranked or missing. Read the RRF row.

In [ ]:
HEATMAP = {
    "BM25": Q_BM25, "Semantic": Q_SEMANTIC,
    "Linear 0.8/0.2": q_linear(0.8, 0.2),
    "Linear 0.5/0.5": q_linear(0.5, 0.5),
    "Linear 0.2/0.8": q_linear(0.2, 0.8),
    "RRF hybrid": Q_RRF,
}
QUERY_LABELS = [q for q, _, _ in JUDGMENTS]
matrix = [[rank_of(t, q, gid) for q, gid, _ in JUDGMENTS] for t in HEATMAP.values()]
CAP = 8

try:
    import matplotlib.pyplot as plt
    color_vals = [[min(r, CAP) if r else CAP for r in row] for row in matrix]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    im = ax.imshow(color_vals, cmap="RdYlGn_r", vmin=1, vmax=CAP, aspect="auto")
    ax.set_xticks(range(len(QUERY_LABELS))); ax.set_xticklabels(QUERY_LABELS, rotation=20, ha="right")
    ax.set_yticks(range(len(HEATMAP))); ax.set_yticklabels(list(HEATMAP))
    for i, row in enumerate(matrix):
        for j, r in enumerate(row):
            ax.text(j, i, "—" if r is None else str(r), ha="center", va="center", fontweight="bold")
    ax.set_title("Rank of the correct doc — lower (green) is better", pad=12)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(f"rank (capped at {CAP})")
    plt.tight_layout(); plt.show()
except ImportError:
    GREEN, YELLOW, RED, RESET = "\033[42m\033[30m", "\033[43m\033[30m", "\033[41m\033[97m", "\033[0m"
    def cell(r):
        bg = GREEN if r == 1 else (YELLOW if r and r <= 3 else RED)
        return f"{bg} {('—' if r is None else r):>2} {RESET}"
    print(f"{'strategy':<16}" + "".join(f"{q[:13]:<15}" for q in QUERY_LABELS))
    for name, row in zip(HEATMAP, matrix):
        print(f"{name:<16}" + "".join(f"  {cell(r)}        "[:15] for r in row))

print("\nRRF row: green across every query. Every other row has at least one non-green cell.")

## Precision after recall — the `RERANK` command

FUSE gives you strong **recall** (the right docs are in the top-N). A reranker adds **precision** — a cross-encoder re-scores the top candidates against the query and reorders them. In ES|QL it's a pipe stage that rewrites `_score`:

```esql
... | FUSE | LIMIT 20
| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}
| LIMIT 5
```

Note the `LIMIT 20` before `RERANK`: you hand the reranker a real candidate set (reranking 1 doc is a no-op), then keep the top few after it reorders. We'll go deeper on rerankers in Lab 5.

In [ ]:
Q_RERANK = (
    "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
    "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
    "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
    "| FUSE | SORT _score DESC | LIMIT 20\n"
    '| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}\n'
    "| SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score"
    # ^ The SORT _score DESC AFTER RERANK is REQUIRED. RERANK overwrites _score but does
    #   NOT reorder rows — without this sort, rows keep their pre-rerank (RRF) order and
    #   the reranking is computed but invisible.
)
q = "reduce storage cost for old logs"   # RRF puts doc-041 #1; the v3 reranker promotes doc-017
try:
    print(f"RRF recall stage: {q!r}")
    show_esql(esql(Q_RRF, q=q))
    print(f"\nAfter RERANK (listwise precision): {q!r}")
    show_esql(esql(Q_RERANK, q=q))
    print("\n(_score is now the reranker's relevance score, and the order changed because")
    print(" we re-sorted on it — note RERANK alone does not reorder.)")
except Exception as e:
    print(f"⚠ RERANK unavailable: {type(e).__name__}: {str(e)[:160]}")
    print("  Verify '.jina-reranker-v3' exists in es.inference.get(). RRF above is still production-grade.")

## Decision framework — which retriever for which situation?

| | RRF (`FUSE`) | Linear (`FUSE LINEAR`) |
|---|---|---|
| Normalization needed | No (rank-based) | Yes (`minmax`) |
| Weight tuning | None | Per branch |
| Recalibration when corpus changes | Not needed | Required |
| Production default | ✅ Yes | When you've *measured* the weights |

**RRF works out of the box and stays stable** as the corpus grows or the embedding model changes. Linear can outperform RRF *if* you've measured the right weights for your data and query mix — but those weights go stale. Add **`RERANK`** on top when the order at the very top must be exactly right.

**Next:** wire this retriever to an LLM — and prove that retrieval quality, not model quality, sets the answer ceiling. Lab 4.

---
*Continue in Discover → Lab 4 assignment, or open `lab4-esql-rag-pipeline.ipynb`*